# Installation

In [3]:
#!pip uninstall -y transformers torch torchvision

In [5]:
#!pip install git+https://github.com/dnth/rag-datakit.git

In [1]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import BatchSamplers
from datasets import load_dataset

In [2]:
dataset = load_dataset("frankwong2001/ssf-train-valid-full-synthetic-batch10")
dataset

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 4524
    })
    valid: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 1131
    })
})

In [3]:
dataset['valid'][0]

{'anchor': 'The Assistant Equipment Engineer applies engineering principles and techniques to support equipment engineering processes in a manufacturing environment to meet organisational objectives. He/She also assists in analysing equipment maintenance issues. In addition, the Assistant Equipment Engineer participates in equipment improvement projects, and partakes in the development of maintenance plans in accordance with organisational objectives. The Assistant Equipment Engineer is required to have strong communication skills, good teamwork and an analytical mind to perform his role well to achieve the desired organisational outcomes.',
 'positive': 'The Assistant Equipment Engineer utilizes engineering principles and techniques to enhance equipment engineering processes within a manufacturing setting, aligning with organizational goals. He/She also aids in evaluating equipment maintenance challenges. Furthermore, the Assistant Equipment Engineer engages in equipment enhancement i

# W&B and Model Configuration

In [4]:
import wandb
import os
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Fetch the WANDB_API_KEY from the environment
wandb_api_key = os.getenv("WAB_API_KEY")

# Log in using the API key
wandb.login(key=wandb_api_key)

model_id = "sentence-transformers/all-minilm-l6-v2"
save_model_path = "./models/all-minilm-l6-v2"

wandb.init(project="rag-datakit-finetunes", name="all-minilm-l6-v2~frankwong2001/ssf-train-valid-full-synthetic-batch10")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/frank123/.netrc
wandb: Currently logged in as: frankwong2001 (frankwong2001-cxsanalytics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Training Arguments

In [5]:
args = SentenceTransformerTrainingArguments(
    output_dir=save_model_path,
    num_train_epochs=5,                         # number of epochs
    per_device_train_batch_size=32,             # train batch size
    gradient_accumulation_steps=16,             # for a global batch size of 512
    per_device_eval_batch_size=16,              # evaluation batch size
    warmup_ratio=0.1,                           # warmup ratio
    learning_rate=2e-5,                         # learning rate, 2e-5 is a good value
    lr_scheduler_type="cosine",                 # use cosine learning rate scheduler
    optim="adamw_torch_fused",                  # use fused adamw optimizer
    tf32=False,                                  # use tf32 precision
    bf16=True,                                  # use bf16 precision
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
    eval_strategy="epoch",                      # evaluate after each epoch
    save_strategy="epoch",                      # save after each epoch
    logging_strategy="epoch",                   # log after each epoch
    save_total_limit=3,                         # save only the last 3 models
    load_best_model_at_end=True,                # load the best model when training ends
    report_to="wandb"
    )

In [6]:
model = SentenceTransformer(model_id)
train_loss = MultipleNegativesRankingLoss(model)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['valid'],  
    loss=train_loss,
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

# Execute Training


In [7]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.025100,0.003800
2,0.006400,0.001880
3,0.003100,0.001486
4,0.003300,0.001358
5,0.003300,0.001343


TrainOutput(global_step=45, training_loss=0.008226948355635007, metrics={'train_runtime': 139.6745, 'train_samples_per_second': 161.948, 'train_steps_per_second': 0.322, 'total_flos': 0.0, 'train_loss': 0.008226948355635007, 'epoch': 5.0})

#  Save & Upload Model

In [8]:
trainer.save_model()

In [9]:
import os
wandb.save(os.path.join(save_model_path, "*"))

wandb: WARNING Symlinked 16 files into the W&B run directory, call wandb.save again to sync new files.


['/home/frank123/GitHub/Rag-datakit/rag-datakit/nbs-frank/wandb/run-20250829_180610-481zz5f8/files/models/all-minilm-l6-v2/modules.json',
 '/home/frank123/GitHub/Rag-datakit/rag-datakit/nbs-frank/wandb/run-20250829_180610-481zz5f8/files/models/all-minilm-l6-v2/model.safetensors',
 '/home/frank123/GitHub/Rag-datakit/rag-datakit/nbs-frank/wandb/run-20250829_180610-481zz5f8/files/models/all-minilm-l6-v2/checkpoint-36',
 '/home/frank123/GitHub/Rag-datakit/rag-datakit/nbs-frank/wandb/run-20250829_180610-481zz5f8/files/models/all-minilm-l6-v2/checkpoint-27',
 '/home/frank123/GitHub/Rag-datakit/rag-datakit/nbs-frank/wandb/run-20250829_180610-481zz5f8/files/models/all-minilm-l6-v2/config_sentence_transformers.json',
 '/home/frank123/GitHub/Rag-datakit/rag-datakit/nbs-frank/wandb/run-20250829_180610-481zz5f8/files/models/all-minilm-l6-v2/tokenizer.json',
 '/home/frank123/GitHub/Rag-datakit/rag-datakit/nbs-frank/wandb/run-20250829_180610-481zz5f8/files/models/all-minilm-l6-v2/tokenizer_config.js

In [10]:
wandb.finish()

eval/loss,█▃▁▁▁
eval/runtime,▁▆▃█▅
eval/samples_per_second,█▃▆▁▄
eval/steps_per_second,█▃▆▁▄
train/epoch,▁▁▃▃▅▅▆▆███
train/global_step,▁▁▃▃▅▅▆▆███
train/grad_norm,█▂▃▁▆
train/learning_rate,█▇▄▂▁
train/loss,█▂▁▁▁
eval/loss,0.00134
eval/runtime,3.0003


# Push to Hugging Face

In [11]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import Trainer

# Load environment variables from .env file
load_dotenv()

# Fetch the Hugging Face API key from the environment
hf_api_key = os.getenv("HF_TOKEN")

# Log in using the Hugging Face API key
login(token=hf_api_key)

# Assuming you have a Trainer object `trainer`
trainer.model.push_to_hub("frankwong2001/1_attempt_all-minilm-l6-v2", exist_ok=True)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmptd_vpjrf/model.safetensors    :   3%|3         | 2.84MB / 90.9MB            

'https://huggingface.co/frankwong2001/1_attempt_all-minilm-l6-v2/commit/9a8fb10a2622bbdf6204685e6d139b7a770abfc8'